In [2]:
from email import utils
import polars as pl
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon, MultiPolygon, Point
import matplotlib.pyplot as plt
from pathlib import Path
import sys
sys.path.append("../src")  # Add path to where 'retail' package lives
import retail.retailfilter as rf # Correct import path
import retail.retailfilter_new as rfn
import utils.utils as ut  # Correct import path


In [3]:

# Input file
# input_file = Path("../data/input_files/uk_foursquare_datefields_admin_categories.parquet")
input_file = Path("../data/input_files/uk_foursquare_cc_aug.parquet")
# Read in parquet
foursquare = pl.read_parquet(input_file)
print(len(foursquare))
# Filter
foursquare_filtnew = rfn.filter_foursquare(foursquare)
print(len(foursquare_filtnew))



# Columns to keep
COLUMNS_TO_KEEP = [
    "fsq_place_id", "name", "lat", "lng", "address", "locality", "region",
    "postcode", "admin_region", "post_town", "po_box", "country",
    "date_created", "date_refreshed", "date_closed", "tel", "website", 
    "fsq_category_ids", "fsq_category_labels", "h3_11"
]

# name has changed now its exploded.
if "fsq_category_ids" in foursquare_filtnew.columns:
    COLUMNS_TO_KEEP_MOD = [col if col != "fsq_category_id" else "fsq_category_ids" for col in COLUMNS_TO_KEEP]
elif "fsq_category_id" in foursquare_filtnew.columns:
    COLUMNS_TO_KEEP_MOD = [col if col != "fsq_category_ids" else "fsq_category_id" for col in COLUMNS_TO_KEEP]
else:
    COLUMNS_TO_KEEP_MOD = COLUMNS_TO_KEEP

# Add H3 column
foursquare_filtnew = ut.add_h3_11_column(foursquare_filtnew)

# Keep only selected columns
foursquare_filtnew = foursquare_filtnew.select(COLUMNS_TO_KEEP_MOD)





4685463
Filtered Foursquare data (Retail Cats): 1919435 rows remaining.
Filtered Foursquare data (remove post box): 1919382 rows remaining.
Filtered Foursquare data (remove closed): 1661090 rows remaining.
1492824


In [15]:
defunct_retailers = pl.read_csv("../data/input_files/defunct_uk_retailers_wikipedia_full.csv")


# 1) Drop repeats on Brand column
defunct_retailers = defunct_retailers.unique(subset=["Brand"])

# 2) Remove anything in brackets () from Brand
defunct_retailers = defunct_retailers.with_columns(
    pl.col("Brand").str.replace(r"\s*\(.*?\)", "", literal=False)
)


In [18]:


# Assumptions:
# - defunct_retailers: pl.DataFrame with column "Brand"
# - foursquare_filtnew: pl.DataFrame with column "name"

# 1) Normalise brand names (case/whitespace) for matching
def make_key(col: pl.Expr) -> pl.Expr:
    return col.str.strip_chars().str.to_lowercase()

defunct = defunct_retailers.with_columns(
    brand_key = make_key(pl.col("Brand"))
)

fs = foursquare_filtnew.with_columns(
    brand_key = make_key(pl.col("name"))
)

# 2) Rows in FS that match a defunct brand (to be removed)
removed = fs.join(defunct.select("brand_key").unique(), on="brand_key", how="inner")

# 3) Count how many FS POIs removed per defunct brand
removed_counts = (
    removed
    .group_by("brand_key")
    .len()
    .rename({"len": "removed_poi_count"})
)

# 4) Add the counts back to defunct_retailers (0 if none)
defunct_with_counts = (
    defunct
    .join(removed_counts, on="brand_key", how="left")
    .with_columns(pl.col("removed_poi_count").fill_null(0).cast(pl.Int64))
    .drop("brand_key")
)

# 5) The filtered FS dataframe (with defunct brands removed)
foursquare_filtered = fs.join(
    defunct.select("brand_key").unique(),
    on="brand_key",
    how="anti"
).drop("brand_key")


In [23]:
defunct_with_counts = defunct_with_counts.sort("removed_poi_count", descending=True)

In [ ]:
defunct_with_counts

,Brand,WikipediaListSource,SourceURL
0,...instore,Defunct discount shops (per list page),https://en.wikipedia.org/wiki/List_of_discount...
2,99p Stores,Defunct discount shops (per list page),https://en.wikipedia.org/wiki/List_of_discount...
4,Adams Kids,Defunct retail companies of the United Kingdom,https://en.wikipedia.org/wiki/Category:Defunct...
5,Afflecks,Defunct department stores of the United Kingdom,https://en.wikipedia.org/wiki/Category:Defunct...
7,Alldays,Defunct supermarkets of the United Kingdom,https://en.wikipedia.org/wiki/Category:Defunct...
...,...,...,...
436,Woolworths,Defunct department stores of the United Kingdom,https://en.wikipedia.org/wiki/Category:Defunct...
438,Wrensons Stores,Defunct supermarkets of the United Kingdom,https://en.wikipedia.org/wiki/Category:Defunct...
439,Wright's Biscuits,Defunct retail companies of the United Kingdom,https://en.wikipedia.org/wiki/Category:Defunct...
440,Zacharias and Co.,Defunct retail companies of the United Kingdom,https://en.wikipedia.org/wiki/Category:Defunct...


In [ ]:
import polars as pl
import geopandas as gpd
import re

# -----------------------------
# Assumptions from earlier:
# - defunct_retailers: pl.DataFrame with "Brand"
# - foursquare_filtnew: pl.DataFrame with "name", "lat", "lng"
# -----------------------------

# Helper to normalize for matching
def make_key(col: pl.Expr) -> pl.Expr:
    return col.str.strip_chars().str.to_lowercase()

# 0) (Optional) Clean + dedupe defunct list like before
defunct_retailers = (
    defunct_retailers
    .unique(subset=["Brand"])
    .with_columns(
        # remove bracketed suffixes e.g. "Woolworths (UK)" -> "Woolworths"
        pl.col("Brand").str.replace(r"\s*\(.*?\)", "", literal=False)
    )
)

# 1) Build normalized keys
defunct = defunct_retailers.with_columns(brand_key = make_key(pl.col("Brand")))
fs = foursquare_filtnew.with_columns(brand_key = make_key(pl.col("name")))

# 2) Find rows to remove (the removed POIs)
removed = fs.join(defunct.select("brand_key").unique(), on="brand_key", how="inner")

# 3) Top 10 brands by number of removed POIs
removed_counts = (
    removed
    .group_by("brand_key")
    .len()
    .rename({"len": "removed_poi_count"})
    .sort("removed_poi_count", descending=True)
)

top10_keys = removed_counts.select("brand_key").head(10)

# 4) Keep only removed POIs for those top 10 brands
top10_removed = removed.join(top10_keys, on="brand_key", how="inner")

# 5) Add human-readable brand back in
defunct_lookup = defunct.select(["brand_key", "Brand"]).unique()
top10_removed = (
    top10_removed
    .join(defunct_lookup, on="brand_key", how="left")
    .rename({"Brand": "defunct_brand"})
)

# 6) Ensure lat/lng present
top10_removed = top10_removed.filter(
    pl.all_horizontal(
        pl.col("lat").is_not_null(),
        pl.col("lng").is_not_null()
    )
)

# 7) Write one GeoParquet per brand of removed locations
out_dir = Path("defunct_removed_top10")
out_dir.mkdir(parents=True, exist_ok=True)

# Partition by brand
parts = top10_removed.partition_by("defunct_brand", as_dict=False)

for i, df in enumerate(parts, start=1):
    pdf = df.to_pandas()
    gdf = gpd.GeoDataFrame(
        pdf,
        geometry=gpd.points_from_xy(pdf["lng"], pdf["lat"]),
        crs="EPSG:4326"
    )
    fname = out_dir / f"topbrand{i}.geo.parquet"
    gdf.to_parquet(fname)
    print(f"Wrote {fname.name} ({len(gdf)} points)")


AttributeError: 'tuple' object has no attribute 'to_pandas'

In [31]:
#search for woolworths in foursquare_filtered
woolworths_fs = foursquare_filtered.filter(pl.col("name").str.contains("woolworth"))

In [32]:
woolworths_fs

fsq_place_id,name,lat,lng,address,locality,region,postcode,admin_region,post_town,po_box,country,date_created,date_refreshed,date_closed,tel,website,fsq_category_id,fsq_category_labels,h3_11
str,str,f64,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,list[str],str
"""4b59c5acf964a520439728e3""","""woolworths social club""",53.603172,-2.176291,null,null,null,null,null,null,null,"""GB""","""2010-01-22""","""2024-05-02""",null,null,null,"""52e81612bcbc57f1066b7a33""","[""Community and Government > Social Club""]","""8b19424f0335fff"""
